# WET-003: Donut Vignetting Test

This notebook will examine the effect that using vignetted sources has on the wavefront corrections calculated by the Rubin AOS pipeline.

Owner: **Bryce Kalmbach** <br>
Last Verified to Run: **2025-03-28** <br>
Software Version:
  - `ts_wep`: **14.1.1** 
  - `lsst_distrib`: **w_2025_13**

In [ ]:
# Times Square Parameters
collection_name = 'u/brycek/aosRefitWcs_danish'
detector = 0
min_seq_num = 90
max_seq_num = 107
day_obs = 20241115

In [ ]:
import os
from copy import copy

import numpy as np
from astropy.visualization import ZScaleInterval
from matplotlib import pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap, BoundaryNorm
from ipywidgets import interact, IntSlider
import ipywidgets as widgets
from IPython.display import display
%matplotlib widget

In [ ]:
from lsst.daf.butler import Butler
from lsst.afw.cameraGeom import FIELD_ANGLE

In [ ]:
butler = Butler('/repo/main')

In [ ]:
camera = butler.get('camera', {'instrument': "LSSTComCam"}, collections=collection_name)

## Define useful functions

These are helpful functions that enable interactive plotting or calculate values of interest. 
Since Times Square cannot read functions from within the repository we put them in the notebook.
When running on Times Square the option to avoid seeing code cells make these invisible to the regular user.

In [ ]:
def calc_mean_field_distance(table_in):
    """Average field distance between intra and extra exposures"""
    field_distance = 0.5 * (np.sqrt(table_in['thx_CCS_intra']**2 + table_in['thy_CCS_intra']**2) + np.sqrt(table_in['thx_CCS_extra']**2 + table_in['thy_CCS_extra']**2))
    return field_distance

In [ ]:
# Function to display a specific plot
source_index = 0

def display_source_plot(index):
    """Display sources identified in each visit and add ability to scan through the visits interactively."""
    with output:
        output.clear_output(wait=True)
        # plt.close('all')
        fig = plt.figure(figsize=(20, 8))
        gs = GridSpec(4, 9, figure=fig)
        ax_main = fig.add_subplot(gs[:, :4])
        data = plot_data[index]
        ax_main.scatter(np.degrees(data['x']), np.degrees(data['y']), c=data['c'])
        ax_main.scatter(np.degrees(data['closest_x']), np.degrees(data['closest_y']), label='Closest', marker='x', s=200, c='C0')
        ax_main.scatter(np.degrees(data['furthest_x']), np.degrees(data['furthest_y']), label='Furthest', marker='x', s=200, c='C1')
        ax_main.set_title(f'Corner Donuts in Visit: {data['visit']}')
        ax_main.set_xlabel('Field Angle X (degrees)')
        ax_main.set_ylabel('Field Angle Y (degrees)')
        # plt.grid(True)
        # plt.ylim(-1.5, 1.5)

        for i in range(9):
            det = camera[i]
            corners = det.getCorners(FIELD_ANGLE)
            corner_arr = []
            for corner in corners:
                corner_arr.append([np.degrees(corner.getX()), np.degrees(corner.getY())])
            cent = det.getCenter(FIELD_ANGLE)
            poly = Polygon(corner_arr, fill=False)
            ax_main.add_patch(poly)
            ax_main.text(np.degrees(cent.getX()), np.degrees(cent.getY()), f'{det.getName()}', ha='center')
        
        ax_main.legend()

        closest_src = visit_dict[data['visit']]['closest']
        furthest_src = visit_dict[data['visit']]['furthest']
        avg_table = visit_dict[data['visit']]['avg_table']
        for i, det_name in enumerate(corner_names):
            ax_side = fig.add_subplot(gs[i, 4])
            ax_side.imshow(closest_src[det_name]['donut_stamp_intra'].stamp_im.image.array)
            ax_side.set_title(f'{det_name}. {np.degrees(closest_src[det_name]['field_distance']):.3f} deg')
            ax_side = fig.add_subplot(gs[i, 5])
            ax_side.imshow(furthest_src[det_name]['donut_stamp_intra'].stamp_im.image.array)
            ax_side.set_title(f'{det_name}. {np.degrees(furthest_src[det_name]['field_distance']):.3f} deg')
            ax_side = fig.add_subplot(gs[i, 6])
            ax_side.imshow(closest_src[det_name]['donut_stamp_intra'].stamp_im.mask.array - furthest_src[det_name]['donut_stamp_intra'].stamp_im.mask.array)
            ax_side.set_title('Model Diff (Close - Far)')
            ax_side = fig.add_subplot(gs[i, 7:])
            ax_side.plot(zn_selected, closest_src[det_name]['zern_ccs'], label='Closest')
            ax_side.plot(zn_selected, furthest_src[det_name]['zern_ccs'], label='Furthest')
            ax_side.plot(zn_selected, avg_table[avg_table['detector'] == det_name]['zk_CCS'][0], label='Detector Average')
            ax_side.legend()
            ax_side.set_title(f'Zernike Wavefront Estimates for {det_name}.')
        plt.tight_layout()
                                      
        plt.show()
    index_label.value = f'Plot: {index+1}/{num_plots}'

# Define button click handlers
def on_prev_button_clicked(b):
    global source_index
    source_index = max(0, source_index - 1)
    display_source_plot(source_index)

def on_next_button_clicked(b):
    global source_index
    source_index = min(len(plot_data) - 1, source_index + 1)
    display_source_plot(source_index)

## Gather sources

Get the visits that have results from wavefront estimation by checking the visits that have the dataset type `aggregateAOSVisitTableAvg`.

In [ ]:
refs_with_visit_tables = butler.query_datasets('aggregateAOSVisitTableAvg', 
                                               collections=collection_name,
                                               where=f"exposure.day_obs = {day_obs} and exposure.seq_num >= {min_seq_num} and exposure.seq_num <= {max_seq_num} and instrument = 'LSSTComCam'")

In [ ]:
visit_data_id_dict = {x.dataId['visit']: x.dataId.to_simple().dataId for x in refs_with_visit_tables}
visit_list = [x.dataId['visit'] for x in refs_with_visit_tables]
print(f'Gathering results from the following visits: \n{visit_list}')

In [ ]:
zn_selected = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 21, 22, 27, 28]
corner_names = ['R22_S00', 'R22_S02', 'R22_S20', 'R22_S22']
print(f'Estimating wavefronts with the following Zernikes: {zn_selected}')

Compile the average estimated wavefront for each detector from each visit. 
Also save the individual wavefront estimates for the closest and furthest sources identified on the edge sensors.

In [ ]:
visit_dict = dict()
plot_data = list()
for visit in visit_list:
    print(f'Compiling data for Visit: {visit}')
    dataId = visit_data_id_dict[visit]
    raw_table = butler.get('aggregateAOSVisitTableRaw', dataId=dataId, collections=collection_name)
    avg_table = butler.get('aggregateAOSVisitTableAvg', dataId=dataId, collections=collection_name)
    field_distance = calc_mean_field_distance(raw_table)
    raw_table['field_distance'] = field_distance
    
    closest_src = {}
    furthest_src = {}
    visit_x = []
    visit_y = []
    visit_dist = []
    closest_x = []
    closest_y = []
    furthest_x = []
    furthest_y = []
    for det_name in corner_names:
        print(det_name)
        det_idx = np.where(raw_table['detector'] == det_name)
        det_table = raw_table[det_idx]
        
        det_obj = camera[det_name]
        det_dataId = copy(dataId)
        det_dataId['detector'] = det_obj.getId()
    
        # Only get data from donuts used in the final average (remove outliers)
        zern_metadata = butler.get('calcZernikesTask_metadata', dataId=det_dataId, collections=collection_name)
        keep_list = ~np.bool(zern_metadata['calcZernikesTask:combineZernikes'].arrays['combineZernikesFlags'])
        det_table = det_table[keep_list]
        keep_idx = np.where(keep_list == True)[0]
    
    
        closest_idx = np.argmin(det_table['field_distance'])
        furthest_idx = np.argmax(det_table['field_distance'])
        closest_x.append(det_table[closest_idx]['thx_CCS_intra'])
        closest_y.append(det_table[closest_idx]['thy_CCS_intra'])
        furthest_x.append(det_table[furthest_idx]['thx_CCS_intra'])
        furthest_y.append(det_table[furthest_idx]['thy_CCS_intra'])
        ds_intra = butler.get('donutStampsIntra', dataId=det_dataId, collections=collection_name)
        ds_extra = butler.get('donutStampsExtra', dataId=det_dataId, collections=collection_name)
        mask_dict = ds_intra[keep_idx[furthest_idx]].stamp_im.mask.getMaskPlaneDict()
        vignette_frac = np.sum(ds_intra[keep_idx[furthest_idx]].stamp_im.mask.array == 2**mask_dict['DONUT']) / np.sum(ds_intra[keep_idx[closest_idx]].stamp_im.mask.array == 2**mask_dict['DONUT'])
    
        closest_src[det_name] = {
            'idx': closest_idx, 
            'field_distance': det_table[closest_idx]['field_distance'], 
            'zern_ccs': det_table[closest_idx]['zk_CCS'], 
            'donut_stamp_intra': ds_intra[keep_idx[closest_idx]], 
            'donut_stamp_extra': ds_extra[keep_idx[closest_idx]]
        }
        furthest_src[det_name] = {
            'idx': furthest_idx, 
            'field_distance': det_table[furthest_idx]['field_distance'], 
            'zern_ccs': det_table[furthest_idx]['zk_CCS'], 
            'donut_stamp_intra': ds_intra[keep_idx[furthest_idx]], 
            'donut_stamp_extra': ds_extra[keep_idx[furthest_idx]],
            'vignette_frac': vignette_frac,
        }
        visit_x.append(det_table['thx_CCS_intra'].value)
        visit_y.append(det_table['thy_CCS_intra'].value)
        visit_dist.append(det_table['field_distance'].value)
    visit_dict[visit] = dict()
    visit_dict[visit]['avg_table'] = avg_table[['detector', 'zk_CCS']]
    visit_dict[visit]['closest'] = closest_src
    visit_dict[visit]['furthest'] = furthest_src
    visit_dict[visit]['rot_angle'] = np.degrees(raw_table.meta['rotAngle'])

    plot_data.append({
        'x': np.concatenate(visit_x),
        'y': np.concatenate(visit_y),
        'c': np.concatenate(visit_dist),
        'closest_x': closest_x, 
        'closest_y': closest_y,
        'furthest_x': furthest_x, 
        'furthest_y': furthest_y,
        'visit': visit,
    })

Below we create a summary plot showing the locations of the donuts on the corner sensors as well as the closest and furthest donuts for each sensor and their wavefront estimates compared to the average for each sensor.

In [ ]:
# Create output widget to display the plot
output = widgets.Output()
num_plots = len(plot_data)

# Create navigation buttons
prev_button = widgets.Button(description='Previous')
next_button = widgets.Button(description='Next')
index_label = widgets.Label(value=f'Plot: 1/{num_plots}')

prev_button.on_click(on_prev_button_clicked)
next_button.on_click(on_next_button_clicked)

controls = widgets.HBox([prev_button, index_label, next_button])
ui = widgets.VBox([controls, output])

display(ui)
display_source_plot(0)

### Calculate Corrections

In this section we use the wavefront estimates from above to calculate the corrections on the optical system using the `ts_ofc` package.
We calculate the corrections for the average wavefront estimates across the focal plane and then compare it to situations where we replace the average value on one of the corner sensors with the individual wavefront of the furthest (most vignetted) donut source on that sensor.
We compare the results for these two situations across all visits and in each visit have four iterations where we replace on of the corner sensors in this way.
We also did the same where we compare the same procedure but with the closest donuts which should have no vignetting as a control.

In [ ]:
from lsst.ts.ofc import OFC, OFCData
ofc_data = OFCData('comcam')
ofc_data.controller_filename = os.path.join(os.environ['TS_OFC_DIR'], 'policy', 'configurations', 'oic_controller.yaml')

ofc_calc = OFC(ofc_data)

sensor_ids = [camera.getNameMap()[name].getId() for name in avg_table['detector']]
ofc_calc.ofc_data.znmin = 4
ofc_calc.ofc_data.znmax = 28
ofc_calc.ofc_data.zn_selected = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 21, 22, 27, 28])
zn_idx_array = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 21, 22, 27, 28]) - 4

In [ ]:
ofc_calc.reset()
furthest_corrections_all = []
furthest_distance_all = []
furthest_vignetting_all = []
closest_corrections_all = []
closest_distance_all = []
average_corrections_all = []
det_list = []
det_name_dict = {'R22_S00': 0, 'R22_S20': 1, 'R22_S22': 2, 'R22_S02': 3}
for visit in visit_list:
    print(visit)
    ofc_calc.reset()
    rot_angle = visit_dict[visit]['rot_angle']
    avg_table = visit_dict[visit]['avg_table']
    furthest_src = visit_dict[visit]['furthest']
    use_array = np.zeros((len(sensor_ids), 25))
    use_array[:, zn_idx_array] = avg_table['zk_CCS']
    
    ofc_calc.calculate_corrections(
        wfe=use_array,
        sensor_ids=sensor_ids,
        filter_name='r',
        rotation_angle=rot_angle
    )
    average_corrections = ofc_calc.lv_dof
    for det_name in corner_names:
        ofc_calc.reset()
        furthest_avg_table = copy(avg_table)
        use_array = np.zeros((len(sensor_ids), 25))
        det_idx = np.where(furthest_avg_table['detector'] == det_name)[0]
        furthest_avg_table['zk_CCS'][det_idx] = furthest_src[det_name]['zern_ccs']
        use_array[:, zn_idx_array] = furthest_avg_table['zk_CCS']
        
        ofc_calc.calculate_corrections(
            wfe=use_array,
            sensor_ids=sensor_ids,
            filter_name='r',
            rotation_angle=rot_angle
        )
        
        furthest_corrections = ofc_calc.lv_dof
        furthest_corrections_all.append(furthest_corrections)
        furthest_distance_all.append(furthest_src[det_name]['field_distance'])
        furthest_vignetting_all.append(furthest_src[det_name]['vignette_frac'])
    
        ofc_calc.reset()
        closest_avg_table = copy(avg_table)
        use_array = np.zeros((len(sensor_ids), 25))
        closest_avg_table['zk_CCS'][det_idx] = closest_src[det_name]['zern_ccs']
        use_array[:, zn_idx_array] = closest_avg_table['zk_CCS']
        
        ofc_calc.calculate_corrections(
            wfe=use_array,
            sensor_ids=sensor_ids,
            filter_name='r',
            rotation_angle=rot_angle
        )
        
        closest_corrections = ofc_calc.lv_dof
        closest_corrections_all.append(closest_corrections)
        closest_distance_all.append(closest_src[det_name]['field_distance'])        
        
        average_corrections_all.append(average_corrections)
        det_list.append(det_name_dict[det_name])
furthest_corrections_all = np.array(furthest_corrections_all)
closest_corrections_all = np.array(closest_corrections_all)
average_corrections_all = np.array(average_corrections_all)

In [ ]:
from ipywidgets import interact, widgets
# %matplotlib inline

def create_interactive_scatter():
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(16, 12))
    fig.clear()
    x = np.degrees(furthest_distance_all)
    
    categories = corner_names
    colors = ['C0', 'C1', 'C2', 'C3']
    # Create a ListedColormap
    cmap = ListedColormap(colors)
    # Create boundaries for the colormap
    bounds = np.arange(len(categories) + 1) - 0.5
    norm = BoundaryNorm(bounds, cmap.N)
    
    # Create custom legend handles
    legend_handles = [
        plt.scatter([], [], c=color, label=f'{cat}') 
        for cat, color in zip(categories, colors)
    ]
    
    dropdown = widgets.Dropdown(
        options=['Hexapods', 'M1M3', 'M2'],
        value='Hexapods',
        description='Corrections:',
        disabled=False,
    )
    # match dropdown to plot info
    dropdown_plot_dict = {'Hexapods': {'num_plots': 10, 'starting_index': 0}, 'M1M3': {'num_plots': 20, 'starting_index': 10}, 'M2': {'num_plots': 20, 'starting_index': 30}}
    
    def update(change):
        fig.clear()
        
        # Get number of subplots
        num_plots = dropdown_plot_dict[dropdown.value]['num_plots']
        
        # Create subplot grid
        gs = fig.add_gridspec(int(np.ceil(num_plots/4)), 4)
        
        # Get current correction column
        dropdown_index = dropdown_plot_dict[dropdown.value]['starting_index']
        
        # Create subplots
        for i in range(num_plots):
            # Create subplot
            ax = fig.add_subplot(gs[int(np.floor(i/4)), int(i % 4)])
            y = np.abs((furthest_corrections_all[:,dropdown_index] - average_corrections_all[:, dropdown_index]) / average_corrections_all[:, dropdown_index])
            ax.scatter(x, y, c=det_list, cmap=cmap, norm=norm)
            ax.set_xlabel('Field Distance (degrees)')
            ax.set_ylabel('(Avg - Furthest)/Avg')
            ax.set_title(f'DOF #{dropdown_index+1} / 50')

            dropdown_index += 1
        fig.suptitle(f'Corrections for {dropdown.value}')
        # Add a single legend for the entire figure
        fig.legend(handles=legend_handles, 
                   loc='lower center', 
                   bbox_to_anchor=(0.8, 0.94), 
                   ncol=4,  # Adjust ncol as needed
                   title='Detectors',
                   fancybox=True,
                   framealpha=0.5)
        # plt.tight_layout()
        fig.tight_layout()
        fig.subplots_adjust(top=0.92)
        fig.canvas.draw_idle()
    
    # Connect slider to update function
    dropdown.observe(update, names='value')
    update('Hexapods')
    
    # Combine slider and plot
    return widgets.VBox([dropdown, widgets.Output()])

# In a Jupyter notebook:
plot = create_interactive_scatter()
display(plot)
plt.show()

In [ ]:
from ipywidgets import interact, widgets
# %matplotlib inline

def create_interactive_scatter():
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(16, 12))
    fig.clear()
    x = np.degrees(furthest_distance_all)
    
    categories = ['Furthest', 'Closest']
    colors = ['C0', 'C1']
    # Create a ListedColormap
    cmap = ListedColormap(colors)
    # Create boundaries for the colormap
    bounds = np.arange(len(categories) + 1) - 0.5
    norm = BoundaryNorm(bounds, cmap.N)
    
    # Create custom legend handles
    legend_handles = [
        plt.scatter([], [], c=color, label=f'{cat}') 
        for cat, color in zip(categories, colors)
    ]
    
    dropdown = widgets.Dropdown(
        options=['Hexapods', 'M1M3', 'M2'],
        value='Hexapods',
        description='Corrections:',
        disabled=False,
    )
    # match dropdown to plot info
    dropdown_plot_dict = {'Hexapods': {'num_plots': 10, 'starting_index': 0}, 'M1M3': {'num_plots': 20, 'starting_index': 10}, 'M2': {'num_plots': 20, 'starting_index': 30}}
    
    def update(change):
        fig.clear()
        
        # Get number of subplots
        num_plots = dropdown_plot_dict[dropdown.value]['num_plots']
        
        # Create subplot grid
        gs = fig.add_gridspec(int(np.ceil(num_plots/4)), 4)
        
        # Get current correction column
        dropdown_index = dropdown_plot_dict[dropdown.value]['starting_index']
        
        # Create subplots
        for i in range(num_plots):
            # Create subplot
            ax = fig.add_subplot(gs[int(np.floor(i/4)), int(i % 4)])
            ax.scatter(np.degrees(furthest_distance_all), np.abs((furthest_corrections_all[:,dropdown_index] - average_corrections_all[:, dropdown_index]) / average_corrections_all[:, dropdown_index]), c='C0')#, label='Furthest')
            ax.scatter(np.degrees(closest_distance_all), np.abs((closest_corrections_all[:,dropdown_index] - average_corrections_all[:, dropdown_index]) / average_corrections_all[:, dropdown_index]), c='C1')#, label='Closest')
            #plt.colorbar()
            ax.set_xlabel('Field Distance (degrees)')
            ax.set_ylabel('(Avg - Furthest)/Avg')
            ax.set_title(f'DOF #{dropdown_index+1} / 50')
            dropdown_index += 1
        fig.suptitle(f'Corrections for {dropdown.value}')
        # Add a single legend for the entire figure
        fig.legend(handles=legend_handles,
                   loc='lower center', 
                   bbox_to_anchor=(0.8, 0.94), 
                   ncol=2,  # Adjust ncol as needed
                   title='Replaced Average with:',
                   fancybox=True,
                   framealpha=0.5)
        # plt.tight_layout()
        fig.tight_layout()
        fig.subplots_adjust(top=0.92)
        fig.canvas.draw_idle()
    
    # Connect slider to update function
    dropdown.observe(update, names='value')
    update('Hexapods')
    
    # Combine slider and plot
    return widgets.VBox([dropdown, widgets.Output()])

# In a Jupyter notebook:
plot = create_interactive_scatter()
display(plot)
plt.show()

In [ ]:
from ipywidgets import interact, widgets
# %matplotlib inline

def create_interactive_scatter():
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(16, 12))
    fig.clear()
    x = 1.0 - np.array(furthest_vignetting_all)
    
    categories = ['Furthest', 'Closest']
    colors = ['C0', 'C1']
    # Create a ListedColormap
    cmap = ListedColormap(colors)
    # Create boundaries for the colormap
    bounds = np.arange(len(categories) + 1) - 0.5
    norm = BoundaryNorm(bounds, cmap.N)
    
    # Create custom legend handles
    legend_handles = [
        plt.scatter([], [], c=color, label=f'{cat}') 
        for cat, color in zip(categories, colors)
    ]
    
    dropdown = widgets.Dropdown(
        options=['Hexapods', 'M1M3', 'M2'],
        value='Hexapods',
        description='Corrections:',
        disabled=False,
    )
    # match dropdown to plot info
    dropdown_plot_dict = {'Hexapods': {'num_plots': 10, 'starting_index': 0}, 'M1M3': {'num_plots': 20, 'starting_index': 10}, 'M2': {'num_plots': 20, 'starting_index': 30}}
    
    def update(change):
        fig.clear()
        
        # Get number of subplots
        num_plots = dropdown_plot_dict[dropdown.value]['num_plots']
        
        # Create subplot grid
        gs = fig.add_gridspec(int(np.ceil(num_plots/4)), 4)
        
        # Get current correction column
        dropdown_index = dropdown_plot_dict[dropdown.value]['starting_index']
        
        # Create subplots
        for i in range(num_plots):
            # Create subplot
            ax = fig.add_subplot(gs[int(np.floor(i/4)), int(i % 4)])
            ax.scatter(x, np.abs((furthest_corrections_all[:,dropdown_index] - average_corrections_all[:, dropdown_index]) / average_corrections_all[:, dropdown_index]), c='C0')#, label='Furthest')
            ax.scatter(np.zeros(len(closest_distance_all)), np.abs((closest_corrections_all[:,dropdown_index] - average_corrections_all[:, dropdown_index]) / average_corrections_all[:, dropdown_index]), c='C1')#, label='Closest')
            #plt.colorbar()
            ax.set_xlabel('Vignetting Fraction')
            ax.set_ylabel('(Avg - Furthest)/Avg')
            ax.set_title(f'DOF #{dropdown_index+1} / 50')
            dropdown_index += 1
        fig.suptitle(f'Corrections for {dropdown.value}')
        # Add a single legend for the entire figure
        fig.legend(handles=legend_handles,
                   loc='lower center', 
                   bbox_to_anchor=(0.8, 0.94), 
                   ncol=2,  # Adjust ncol as needed
                   title='Replaced Average with:',
                   fancybox=True,
                   framealpha=0.5)
        # plt.tight_layout()
        fig.tight_layout()
        fig.subplots_adjust(top=0.92)
        fig.canvas.draw_idle()
    
    # Connect slider to update function
    dropdown.observe(update, names='value')
    update('Hexapods')
    
    # Combine slider and plot
    return widgets.VBox([dropdown, widgets.Output()])

# In a Jupyter notebook:
plot = create_interactive_scatter()
display(plot)
plt.show()

Looking across the different corrections it does not seem that with the small amount of vignetting we have in ComCam that there are any consistently noticeable effects upon the corrections calculated when using the most vignetted donuts on the ComCam focal plane.